In [1]:
from pathlib import Path
import csv
import getpass
import hashlib
import hmac
import json
import math
import os
from datetime import datetime


def get_data_dir():
    """Permanent folder for saved files: Google Drive on Colab, otherwise the local folder."""
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        folder = Path("/content/drive/MyDrive/FinanceTracker")
    except ImportError:
        folder = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    folder.mkdir(parents=True, exist_ok=True)
    return folder


DATA_DIR = get_data_dir()

SAMPLE_NAME = "MIT103_A3_Revised_T3_2026_Sample_Transactions.csv"

SAMPLE_CSV = DATA_DIR / SAMPLE_NAME if (DATA_DIR / SAMPLE_NAME).exists() else Path(SAMPLE_NAME)

WORKING_CSV = DATA_DIR / "my_transactions.csv"
ACCOUNTS_FILE = DATA_DIR / "demo_accounts.json"

CSV_FIELDS = ["transaction_id", "date", "category", "description", "amount"]
PBKDF2_ITERATIONS = 100_000


ROLE_PERMISSIONS = {
    "admin": ["add", "display", "search", "delete", "summary"],
    "viewer": ["display", "search", "summary"],
}


class Transaction:
    """A single income/expense record. Raises ValueError if any value is missing or invalid."""

    def __init__(self, transaction_id, date, category, description, amount):
        values = {
            "transaction_id": transaction_id,
            "date": date,
            "category": category,
            "description": description,
            "amount": amount,
        }
        for name, value in values.items():
            if value is None or str(value).strip() == "":
                raise ValueError(f"missing value for '{name}'")

        self.transaction_id = str(transaction_id).strip()
        self.date = str(date).strip()
        self.category = str(category).strip()
        self.description = str(description).strip()

        try:
            datetime.strptime(self.date, "%d/%m/%Y")
        except ValueError:
            raise ValueError(f"invalid date '{self.date}' (expected DD/MM/YYYY)")

        try:
            self.amount = float(amount)
        except (TypeError, ValueError):
            raise ValueError(f"invalid amount '{amount}' for transaction {self.transaction_id}")
        if not math.isfinite(self.amount):
            raise ValueError(f"invalid amount '{amount}' for transaction {self.transaction_id}")

    def __str__(self):
        sign = "+" if self.amount >= 0 else "-"
        return (f"[{self.transaction_id}] {self.date} | {self.category:<10} | "
                f"{self.description:<15} | {sign}${abs(self.amount):.2f}")

    def to_dict(self):
        """Dictionary form used for CSV writing."""
        return {
            "transaction_id": self.transaction_id,
            "date": self.date,
            "category": self.category,
            "description": self.description,
            "amount": f"{self.amount:.2f}",
        }


def read_transactions_csv(file_path):
    """Reads a CSV with csv.DictReader. Malformed rows are skipped with a clear warning."""
    transactions = []
    seen_ids = set()
    skipped = 0
    try:
        with open(file_path, mode="r", newline="", encoding="utf-8-sig") as file:
            reader = csv.DictReader(file)
            missing = [h for h in CSV_FIELDS if h not in (reader.fieldnames or [])]
            if missing:
                print(f"[Error] {file_path} is missing required column(s): {', '.join(missing)}. "
                      "No rows were loaded.")
                return transactions

            for row in reader:
                try:
                    if None in row:
                        raise ValueError("too many columns")
                    t = Transaction(*(row[k] for k in CSV_FIELDS))
                    if t.transaction_id.lower() in seen_ids:
                        raise ValueError(f"duplicate transaction ID '{t.transaction_id}'")
                    seen_ids.add(t.transaction_id.lower())
                    transactions.append(t)
                except ValueError as err:
                    skipped += 1
                    print(f"[Warning] Skipping malformed row on line {reader.line_num}: {err}")
    except (OSError, UnicodeDecodeError, csv.Error) as err:
        print(f"[Error] Could not read {file_path}: {err}")

    if skipped:
        print(f"[Warning] {skipped} malformed row(s) were skipped.")
    return transactions


def load_transactions(filename):
    """Loads the working CSV. On a first run it is created from the sample CSV."""
    file_path = Path(filename)

    if file_path.exists():
        transactions = read_transactions_csv(file_path)
        print(f"[Info] Loaded {len(transactions)} transactions from {file_path}")
        return transactions

    if file_path == WORKING_CSV and SAMPLE_CSV.exists():
        print(f"[Info] {WORKING_CSV} not found. Creating it from the sample dataset...")
        transactions = read_transactions_csv(SAMPLE_CSV)
        save_transactions(transactions, WORKING_CSV)
        print(f"[Info] Loaded {len(transactions)} transactions from {SAMPLE_CSV}")
        return transactions

    print(f"[Warning] File '{file_path}' was not found. Starting with an empty list.")
    print("          Upload the sample CSV to your Drive folder 'FinanceTracker' "
          "(or to the Colab session), then run again.")
    return []


def save_transactions(transactions, filename):
    """Saves all transactions to the working CSV (same five headings). Returns True/False."""
    path = Path(filename)
    temp_path = path.with_name(path.name + ".tmp")
    try:
        with open(temp_path, mode="w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=CSV_FIELDS)
            writer.writeheader()
            for t in transactions:
                writer.writerow(t.to_dict())
        os.replace(temp_path, path)
        print(f"[Success] Saved {len(transactions)} transactions to {path}")
        return True
    except OSError as err:
        print(f"[Error] Could not save transactions to {path}: {err}")
        return False


def make_password_verifier(password):
    """Returns 'salt_hex:hash_hex' using a random salt and PBKDF2-HMAC-SHA256."""
    salt = os.urandom(16)
    pwd_hash = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, PBKDF2_ITERATIONS)
    return salt.hex() + ":" + pwd_hash.hex()


def verify_password(password, stored_verifier):
    """Re-hashes the attempt with the stored salt and compares in constant time."""
    try:
        salt_hex, hash_hex = stored_verifier.split(":")
        salt = bytes.fromhex(salt_hex)
        expected_hash = bytes.fromhex(hash_hex)
    except (ValueError, AttributeError):
        return False
    new_hash = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, PBKDF2_ITERATIONS)
    return hmac.compare_digest(new_hash, expected_hash)


def create_demo_accounts(path=ACCOUNTS_FILE):
    """First-run setup: asks for demo passwords and stores only salted verifiers.

    No password is written in this notebook, so nothing secret is committed to GitHub.
    """
    print("\n[Setup] Creating the fictional demo accounts 'admin' and 'viewer'.")
    print("        Choose any demo passwords. Only a salted hash is stored, never the password.")
    accounts = {}
    for username in ("admin", "viewer"):
        while True:
            password = getpass.getpass(f"Choose a password for '{username}' (min 4 chars): ")
            if len(password) >= 4:
                break
            print("[Error] Password too short.")
        accounts[username] = {"verifier": make_password_verifier(password), "role": username}
    try:
        with open(path, mode="w", encoding="utf-8") as file:
            json.dump(accounts, file, indent=2)
        print(f"[Info] Saved account verifiers to {path} (do NOT commit this file).")
    except OSError as err:
        print(f"[Warning] Could not save {path}: {err}. Accounts are valid for this session only.")
    return accounts


def load_accounts(path=ACCOUNTS_FILE):
    """Loads stored verifiers, or runs first-run setup if the file is missing or invalid."""
    path = Path(path)
    if path.exists():
        try:
            with open(path, mode="r", encoding="utf-8") as file:
                accounts = json.load(file)
            if all(u in accounts and "verifier" in accounts[u] and "role" in accounts[u]
                   for u in ("admin", "viewer")):
                return accounts
            print(f"[Warning] {path} is incomplete. Re-running account setup.")
        except (OSError, json.JSONDecodeError, TypeError):
            print(f"[Warning] {path} is unreadable. Re-running account setup.")
    return create_demo_accounts(path)


def login(accounts=None):
    """Authenticates a user (3 attempts). Returns the role, or None if access is denied."""
    print("\n" + "=" * 45)
    print("      PERSONAL FINANCE TRACKER - LOGIN       ")
    print("=" * 45)

    if accounts is None:
        accounts = load_accounts()

    attempts = 3
    while attempts > 0:
        username = input("Enter username (admin / viewer): ").strip().lower()
        password = getpass.getpass("Enter password: ")

        if username in accounts and verify_password(password, accounts[username]["verifier"]):
            role = accounts[username]["role"]
            print(f"\n[Success] Logged in as: {role.upper()}")
            return role

        attempts -= 1
        print(f"[Error] Invalid username or password. {attempts} attempt(s) remaining.\n")

    print("[Alert] Maximum login attempts exceeded. Access denied.")
    return None


def is_authorised(role, action):
    """Role-based access control check, used inside add/delete as well as the menu."""
    return action in ROLE_PERMISSIONS.get(role, [])


def add_transaction(transactions, role):
    """Adds a validated transaction. Admin only. Returns True if a record was added."""
    if not is_authorised(role, "add"):
        print(f"[Permission Denied] Role '{role}': access denied for Add.")
        return False

    print("\n--- Add New Transaction ---")
    while True:
        trans_id = input("Enter transaction ID (e.g. T009): ").strip()
        if not trans_id:
            print("[Error] Transaction ID cannot be empty.")
        elif any(t.transaction_id.lower() == trans_id.lower() for t in transactions):
            print(f"[Error] Transaction ID '{trans_id}' already exists. IDs must be unique.")
        else:
            break

    while True:
        date = input("Enter date (DD/MM/YYYY): ").strip()
        try:
            datetime.strptime(date, "%d/%m/%Y")
            break
        except ValueError:
            print("[Error] Invalid date. Use DD/MM/YYYY, e.g. 06/11/2026.")

    while True:
        category = input("Enter category (e.g. Food, Transport, Income): ").strip()
        if category:
            break
        print("[Error] Category cannot be empty.")

    while True:
        description = input("Enter description: ").strip()
        if description:
            break
        print("[Error] Description cannot be empty.")

    while True:
        try:
            amount = float(input("Enter amount ($) [positive = income, negative = expense]: "))
            if not math.isfinite(amount):
                raise ValueError
            break
        except ValueError:
            print("[Error] Invalid amount. Please enter a valid number, e.g. -12.50.")

    try:
        transactions.append(Transaction(trans_id, date, category, description, amount))
    except ValueError as err:
        print(f"[Error] Could not create transaction: {err}")
        return False
    print(f"[Success] Transaction {trans_id} added.")
    return True


def display_transactions(transactions):
    """Displays all transactions."""
    print("\n--- All Transactions ---")
    if not transactions:
        print("[!] No transactions recorded.")
        return
    for t in transactions:
        print(t)


def search_transactions(transactions):
    """Searches by transaction ID or category."""
    print("\n--- Search Transactions ---")
    if not transactions:
        print("[!] No transactions stored to search.")
        return

    keyword = input("Enter search keyword (Transaction ID or Category): ").strip().lower()
    if not keyword:
        print("[Error] Search keyword cannot be empty.")
        return

    results = [t for t in transactions
               if keyword in t.transaction_id.lower() or keyword in t.category.lower()]
    if results:
        print(f"\nFound {len(results)} matching transaction(s):")
        for t in results:
            print(t)
    else:
        print(f"[!] No matching transactions found for '{keyword}'.")


def delete_transaction(transactions, transaction_id, role):
    """Deletes a transaction by ID. Admin only. Returns True on success, False otherwise."""
    if not is_authorised(role, "delete"):
        print(f"[Permission Denied] Role '{role}': access denied for Delete.")
        return False

    target_id = str(transaction_id).strip().lower()
    if not target_id:
        print("[Error] Transaction ID cannot be empty.")
        return False

    for index, t in enumerate(transactions):
        if t.transaction_id.lower() == target_id:
            removed = transactions.pop(index)
            print(f"[Success] Removed transaction: {removed}")
            return True

    print(f"[!] Transaction ID '{transaction_id}' not found. List unchanged.")
    return False


def category_totals(transactions):
    """Returns {category: total amount}.

    A dictionary is preferable here because each category is a unique key we update
    in one step (totals[cat] += amount). A list would need a search through the
    existing (category, total) pairs for every transaction.
    """
    totals = {}
    for t in transactions:
        totals[t.category] = totals.get(t.category, 0.0) + t.amount
    return totals


def display_summary(transactions):
    """Displays income, expenses, net balance and dictionary-based category totals."""
    print("\n--- Financial Summary & Category Breakdown ---")
    if not transactions:
        print("[!] No transactions recorded.")
        return

    total_income = sum(t.amount for t in transactions if t.amount > 0)
    total_expense = sum(t.amount for t in transactions if t.amount < 0)
    net_balance = total_income + total_expense

    print(f"Total Income:   +${total_income:.2f}")
    print(f"Total Expense:  -${abs(total_expense):.2f}")
    print("-" * 35)
    print(f"Net Balance:    {'-' if net_balance < 0 else ''}${abs(net_balance):.2f}")

    print("\n[Category Totals]")
    for category, total in category_totals(transactions).items():
        sign = "+" if total >= 0 else "-"
        print(f"  - {category:<12}: {sign}${abs(total):.2f}")


def main():
    """Login, then run the role-based menu until Exit."""
    role = login()
    if not role:
        return

    transactions = load_transactions(WORKING_CSV)

    while True:
        admin_note = "" if is_authorised(role, "add") else "  (Admin only)"
        print("\n" + "=" * 45)
        print(f"   PERSONAL FINANCE TRACKER (Role: {role.upper()})")
        print("=" * 45)
        print(f"1. Add a new transaction{admin_note}")
        print("2. Display all transactions")
        print("3. Search transactions")
        print(f"4. Delete a transaction{admin_note}")
        print("5. View financial summary & category totals")
        print("6. Exit")
        print("=" * 45)

        choice = input("Select an option: ").strip()

        if choice == "1":
            if add_transaction(transactions, role):
                save_transactions(transactions, WORKING_CSV)
        elif choice == "2":
            display_transactions(transactions)
        elif choice == "3":
            search_transactions(transactions)
        elif choice == "4":
            del_id = ""
            if is_authorised(role, "delete"):
                display_transactions(transactions)
                del_id = input("Enter Transaction ID to delete: ").strip()

            if delete_transaction(transactions, del_id, role):
                save_transactions(transactions, WORKING_CSV)
        elif choice == "5":
            display_summary(transactions)
        elif choice == "6":
            print("\nExiting Personal Finance Tracker. Goodbye!")
            break
        else:
            print("[Error] Invalid menu choice. Please enter a number from 1 to 6.")


main()

Mounted at /content/drive

      PERSONAL FINANCE TRACKER - LOGIN       
Enter username (admin / viewer): Admin
Enter password: ··········

[Success] Logged in as: ADMIN
[Info] Loaded 9 transactions from /content/drive/MyDrive/FinanceTracker/my_transactions.csv

   PERSONAL FINANCE TRACKER (Role: ADMIN)
1. Add a new transaction
2. Display all transactions
3. Search transactions
4. Delete a transaction
5. View financial summary & category totals
6. Exit
Select an option: 2

--- All Transactions ---
[T001] 02/11/2026 | Food       | Groceries       | -$48.50
[T002] 03/11/2026 | Transport  | Train card      | -$25.00
[T003] 04/11/2026 | Income     | Casual shift    | +$180.00
[T004] 05/11/2026 | Food       | Lunch           | -$14.20
[T005] 06/11/2026 | Bills      | Mobile plan     | -$35.00
[T006] 07/11/2026 | Transport  | Bus ticket      | -$5.60
[T007] 08/11/2026 | Income     | Refund          | +$20.00
[T008] 09/11/2026 | Food       | Coffee          | -$6.50
[T009] 29/11/2026 | FOOD  